In [1]:
import numpy as np
import nibabel as nib
import SimpleITK as sitk
import os
import glob
from sklearn.metrics import mean_squared_error as MSE
from sklearn.metrics import mean_absolute_error as MAE
import pandas as pd
import json

In [2]:
# vali_folder="/home/changh/Documents/2023_DosePredcition/nnUNet_prediction/HNdose_050_fa"
# label_folder="/home/changh/Documents/2023_DosePredcition/validation/label"
# save_folder="/home/changh/Documents/2023_DosePredcition/validation/diff_50"

vali_folder="/home/changh/Documents/2023_DosePredcition/nnUNet_prediction/HNdose_071_f13e1000_f024e500"
label_folder="/home/changh/Documents/2023_DosePredcition/validation/label"
save_folder="/home/changh/Documents/2023_DosePredcition/validation/diff_71_e1000"
valis=sorted(glob.glob(os.path.join(vali_folder,"*.nii.gz")))
if not os.path.isdir(save_folder):
    os.makedirs(save_folder)

In [3]:
##generate difference map


for vali in valis:
    file=vali.split("/")[-1]
    pred=sitk.ReadImage(os.path.join(vali_folder,file))
    label=sitk.ReadImage(os.path.join(label_folder,file))
    
    pred_np=sitk.GetArrayFromImage(pred)
    label_np=sitk.GetArrayFromImage(label)
    
    sub=label_np-pred_np
    sub_img=sitk.GetImageFromArray(sub)
    sub_img.CopyInformation(pred)
    
    sitk.WriteImage(sub_img,os.path.join(save_folder,str("lb-71_"+file)))
    
    

In [4]:
## calucalte rms json
train_mse=[]
test_mse=[]

train_mae=[]
test_mae=[]

train_error=[]
test_error=[]
cases=[]
for vali in valis:
    case={}
    file=vali.split("/")[-1]
    pred=sitk.ReadImage(os.path.join(vali_folder,file))
    label=sitk.ReadImage(os.path.join(label_folder,file))
    pred_np=sitk.GetArrayFromImage(pred).flatten()
    label_np=sitk.GetArrayFromImage(label).flatten()
    
    
    patient_num=file.split(".")[0]
    patient_dose_error = np.sum(np.abs(label_np - pred_np)) / len(np.flatnonzero(label_np))
    mse=float(MSE(label_np,pred_np))
    mae=float(MAE(label_np,pred_np))
    if int(patient_num)<=240:
        data_type="train"
        train_mse.append(mse)
        train_mae.append(mae)
        train_error.append(patient_dose_error)
    else:
        data_type="test"
        test_mse.append(mse)
        test_mae.append(mae)
        test_error.append(patient_dose_error)
    
    
    case["patient"]=patient_num
    case["type"]=data_type
    case["MSE"]=mse
    case["MAE"]=mae
    case["error"]=patient_dose_error
    cases.append(case)
    
    


In [5]:
dict={}

dict["total_num_of_train"]=len(train_mse)
dict["total_num_of_test"]=len(test_mse)

dict["train_MSE_mean"]=float(np.mean(train_mse))
dict["test_MSE_mean"]=float(np.mean(test_mse))
dict["train_MSE_std"]=float(np.std(train_mse))
dict["test_MSE_std"]=float(np.std(test_mse))

dict["train_MAE_mean"]=float(np.mean(train_mae))
dict["test_MAE_mean"]=float(np.mean(test_mae))
dict["train_MAE_std"]=float(np.std(train_mae))
dict["test_MAE_std"]=float(np.std(test_mae))

dict["train_error_mean"]=float(np.mean(train_error))
dict["test_error_mean"]=float(np.mean(test_error))

dict["cases"]=cases
with open(os.path.join(vali_folder,"metrices.json"), "w") as outfile:
    json.dump(dict, outfile,indent= 4)


/home/changh/anaconda3/envs/nnunet_pytorch/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3474: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/changh/anaconda3/envs/nnunet_pytorch/lib/python3.9/site-packages/numpy/core/_methods.py:189: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
/home/changh/anaconda3/envs/nnunet_pytorch/lib/python3.9/site-packages/numpy/core/_methods.py:264: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/changh/anaconda3/envs/nnunet_pytorch/lib/python3.9/site-packages/numpy/core/_methods.py:222: RuntimeWarning: invalid value encountered in true_divide
  arrmean = um.true_divide(arrmean, div, out=arrmean, casting='unsafe',
/home/changh/anaconda3/envs/nnunet_pytorch/lib/python3.9/site-packages/numpy/core/_methods.py:256: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype

In [6]:
os.path.join(vali_folder,"metrices.json")

'/home/changh/Documents/2023_DosePredcition/nnUNet_prediction/HNdose_071_f13e1000_f024e500/metrices.json'

In [7]:
sorted(test_error)

[1.6044662954565843,
 1.679988389076706,
 1.6804601437159352,
 1.6910227600786687,
 1.7716367096335206,
 1.783514791922242,
 1.8166502515837741,
 1.8166958635746098,
 1.8171942799959635,
 1.8262855616288727,
 1.8313873313067024,
 1.8335661092447695,
 1.859879662116697,
 1.874881623369192,
 1.904933204016592,
 1.9276496950277588,
 1.933488131638536,
 1.9364568279667014,
 1.9469855235036597,
 1.9578213325376017,
 1.9713877281156216,
 2.0014289801256453,
 2.005212462760079,
 2.0739702123383004,
 2.1218841576727896,
 2.1439776285431775,
 2.1479126868334872,
 2.214034682018021,
 2.224065257230782,
 2.2673155520953254,
 2.2700369578479997,
 2.3317100102774466,
 2.3518525682068794,
 2.3889731051344745,
 2.3948374148422515,
 2.456530226647895,
 2.45748952703436,
 2.4754062891687485,
 2.4777423538162484,
 2.4924363493370953,
 2.541394705414013,
 2.5506148382800222,
 2.5588466833003527,
 2.597189900380682,
 2.610528619346043,
 2.639800929911106,
 2.7000566232095045,
 2.752485632550079,
 2.765240